<a href="https://colab.research.google.com/github/Jaychandrarevada/IN226019002_GEN_AI/blob/main/AI_Resume_Screening_System_with_Tracing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U langchain langchain-core langchain-groq langsmith

In [29]:
import os

os.environ["GROQ_API_KEY"] = "your_groq_api_key"

# LangSmith (MANDATORY)
os.environ["LANGCHAIN_API_KEY"] = "your_langsmith_key"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "resume-screening-system"

In [30]:
job_description = """
We are hiring a Data Scientist.

Required Skills:
Python, Machine Learning, Deep Learning, SQL, Data Visualization

Experience: 2+ years

Tools:
TensorFlow, PyTorch, Pandas, Scikit-learn
"""

strong_resume = """
Skills: Python, Machine Learning, Deep Learning, SQL, Data Visualization
Experience: 3 years Data Scientist
Tools: TensorFlow, PyTorch, Pandas, Scikit-learn
"""

average_resume = """
Skills: Python, Machine Learning, SQL
Experience: 1 year Data Analyst
Tools: Pandas, Excel
"""

weak_resume = """
Skills: HTML, CSS
Experience: Fresher
Tools: VS Code
"""

In [46]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",   # ✅ stable + fast + working
    temperature=0
)

In [47]:
extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract:
- Skills
- Experience
- Tools

Return in JSON format.

Resume:
{resume}

STRICT:
Do NOT assume anything.
Return ONLY JSON.
"""
)

extraction_chain = extract_prompt | llm

In [48]:
match_prompt = PromptTemplate(
    input_variables=["resume_data", "job_description"],
    template="""
Compare resume with job description.

Return:
- Matching Skills
- Missing Skills

Resume:
{resume_data}

Job:
{job_description}

STRICT:
Return structured output only.
"""
)

matching_chain = match_prompt | llm

In [49]:
score_prompt = PromptTemplate(
    input_variables=["match_data"],
    template="""
Give a score between 0–100.

Rules:
- More matching skills → higher score
- Missing important skills → lower score

Return ONLY number.

Data:
{match_data}
"""
)

scoring_chain = score_prompt | llm

In [50]:
explain_prompt = PromptTemplate(
    input_variables=["score", "match_data"],
    template="""
Explain the score.

Include:
- Strengths
- Weaknesses

Score: {score}
Data: {match_data}
"""
)

explanation_chain = explain_prompt | llm

In [51]:
def run_pipeline(resume):
    print("\n===== RUNNING PIPELINE =====\n")

    extracted = extraction_chain.invoke({"resume": resume})
    print("EXTRACTED:\n", extracted.content)

    matched = matching_chain.invoke({
        "resume_data": extracted.content,
        "job_description": job_description
    })
    print("\nMATCHED:\n", matched.content)

    score = scoring_chain.invoke({
        "match_data": matched.content
    })
    print("\nSCORE:\n", score.content)

    explanation = explanation_chain.invoke({
        "score": score.content,
        "match_data": matched.content
    })
    print("\nEXPLANATION:\n", explanation.content)

In [52]:
print("===== STRONG =====")
run_pipeline(strong_resume)

print("\n===== AVERAGE =====")
run_pipeline(average_resume)

print("\n===== WEAK =====")
run_pipeline(weak_resume)

===== STRONG =====

===== RUNNING PIPELINE =====

EXTRACTED:
 {
  "Skills": ["Python", "Machine Learning", "Deep Learning", "SQL", "Data Visualization"],
  "Experience": ["3 years Data Scientist"],
  "Tools": ["TensorFlow", "PyTorch", "Pandas", "Scikit-learn"]
}

MATCHED:
 **Matching Skills**
```json
{
  "Matching Skills": ["Python", "Machine Learning", "Deep Learning", "SQL", "Data Visualization"]
}
```

**Missing Skills**
```json
{
  "Missing Skills": []
}
```

Since the resume's experience (3 years) meets the job's experience requirement (2+ years), there are no missing skills based on experience.

SCORE:
 85



EXPLANATION:
 **Score Explanation: 85**

The score of 85 suggests that the candidate's resume is strong in certain areas but has some room for improvement. Here are the strengths and weaknesses:

**Strengths:**

1. **Relevant Technical Skills**: The candidate has a strong set of matching skills, including Python, Machine Learning, Deep Learning, SQL, and Data Visualization. These skills are highly relevant to the job requirements, indicating that the candidate has a solid foundation in the necessary technical areas.
2. **Experience**: The candidate's 3 years of experience meets the job's experience requirement of 2+ years, which is a significant strength.

**Weaknesses:**

1. **Limited Skill Set**: Although the candidate has a strong set of matching skills, there are no additional skills listed in the "Missing Skills" section. This suggests that the candidate may not have a broad range of skills, which could be a limitation in a competitive job market.
2. **Opportunity for Improvement


EXPLANATION:
 **Score Analysis: 40**

The score of 40 indicates a moderate level of match between the candidate's resume and the job description. Here's a breakdown of the strengths and weaknesses:

**Strengths:**

1. **Relevant Skills**: The candidate has listed Python, Machine Learning, and SQL as matching skills, which are highly relevant to the job description. This suggests that the candidate has a solid foundation in the required technical skills.
2. **Partial Experience**: Although the candidate has less than the required 2+ years of experience, they still have some experience in the field, which is a positive aspect.

**Weaknesses:**

1. **Missing Skills**: The candidate is missing several key skills, including Deep Learning, Data Visualization, TensorFlow, PyTorch, and Scikit-learn. These skills are essential for the job, and the candidate's lack of them is a significant weakness.
2. **Insufficient Experience**: The candidate's experience is less than the required 2+ years, w